<!-- RLHF 블로그 설명 괜찮 : https://dalpo0814.tistory.com/56

DPO 이론 :

- https://wikidocs.net/355601
  - 이미지 사용 ( https://wikidocs.net/355600 )
  - SFT는 “이 입력에서 어떤 답을 재현할 것인가?”를 학습합니다. DPO는 같은 프롬프트에 대한 두 답 중 “어느 쪽의 상대 확률을 더 높일 것인가?”를 학습합니다. 두 단계는 대체 관계가 아닙니다. 일반적인 실습 흐름은 먼저 SFT로 과업과 출력 형식을 익히고, 그 체크포인트를 정책 모델과 기준 모델의 출발점으로 삼아 DPO를 수행하는 것입니다
  - 전통적인 보상 모델 기반 RLHF 파이프라인은 선호 데이터로 별도 보상 모델을 학습한 뒤 강화학습으로 정책을 갱신합니다. DPO는 명시적인 보상 모델과 온라인 정책 최적화 단계를 두지 않고 분류형 목적함수로 선호를 학습합니다. 그렇다고 DPO가 항상 더 적은 자원으로 더 좋은 결과를 보장하는 것은 아닙니다. 모델 크기, 기준 모델 처리 방식, 시퀀스 길이와 배치에 따라 메모리 비용이 달라집니다.
-  https://wikidocs.net/236954

DPOTrainer Huggingface : https://huggingface.co/docs/trl/dpo_trainer

- SFT + DPO 예제 코드 : https://wikidocs.net/395248
  - LoRA 어댑터는 베이스 모델과 분리되어 저장됩니다. 추론 시 어댑터를 베이스 모델에 병합하면 추가 오버헤드 없이 사용할 수 있습니다.
  - DPO 훈련 시 주의할 점은 학습률이 SFT보다 더 작다는 점입니다. SFT가 $2 \times 10^{-5}$라면 DPO는 $5 \times 10^{-6}$ 수준이 적절합니다. $\beta$ 파라미터는 일반적으로 0.1을 사용하며, 값이 클수록 참조 모델에서 벗어나지 않으므로 안정적입니다.

- DPOTrainer 내부 collator가 자동으로 completion_mask를 만듭니다. 실제 현재 TRL 소스에서도 prompt 부분에는 0, completion 부분에는 1인 mask를 생성합니다 : https://github.com/huggingface/trl/blob/main/trl/trainer/dpo_trainer.py

- https://wikidocs.net/237188 -> FULL SFT 대신 LoRA + DPO



- Reference model considerations with PEFT : https://huggingface.co/docs/trl/v0.11.1/en/dpo_trainer
  - 3 ways to use ref model -->

<!-- - DPO 유튜브 : https://www.youtube.com/watch?v=IRQL-tu1wDw ( Same author as SFT )

[ THE CHOICE OF REFERENCE IS EVERYTHING 39:45 ]
| Reference Choice | Feasibility | Why It's a Good/Bad Idea |
|---|---|---|
| Base Model | **Poor Choice** | - **Noisy signal** from “statistical parrot”<br>- **Meaningless** comparison baseline |
| SFT Model | **Excellent Choice** | - **Strong baseline** from helpful assistant<br>- **Guardrail** against forgetting SFT training |


[ Reference Score - Video 42:27 ]
$$
r(x, y) = \beta \left( \log \pi_\theta(y \mid x) - \log \pi_{\text{ref}}(y \mid x) \right)
$$

- $\log \pi_\theta(y \mid x)$: The score from our trainable **policy model** (what it *thinks*).

- $\log \pi_{\text{ref}}(y \mid x)$: The score from our frozen **reference model** (what it *used to think*).

- The difference $(\ldots - \ldots)$: The magic. This measures **relative improvement**.


[ Reference Model can aliviate the DPO Problems ]
> Length Bias ( Longer texts are less preferred as the score is sum of negative values )

Reference model can help solve the problem


Winner \($y_w$\) (7 tokens), Loser \($y_l$\) (2 tokens), \($\beta = 0.1$\).

| Response | \( $\log \pi_\theta$ \) (Policy) | \($\log \pi_{\text{ref}}$\) (Reference) | DPO Reward: \($\beta(\log \pi_\theta - \log \pi_{\text{ref}})$\) |
|---|---:|---:|---:|
| \($y_w$\) (Winner) | -0.735 | -0.750 | \($0.1 \times (-0.735 - (-0.750)) = \mathbf{+0.0015}$\) |
| \($y_l$\) (Loser) | -0.210 | -0.200 | \($0.1 \times (-0.210 - (-0.200)) = \mathbf{-0.0010}$\) |


> Bland Prio Bias : just thre reward models give massive scores for common words, for exapmle repetitive words like "the", "like"

reference model already highly probability on these common words


| Optimization Target | $\log \pi_\theta$ (Policy) | $\log \pi_{\text{ref}}$ (Reference) | DPO Reward Gain |
|---|---:|---:|---:|
| Generic word `"the"` | -0.0408 | -0.0513 | $0.1 \times [(-0.0408)-(-0.0513)] = \mathbf{+0.00105}$ |
| Specific word `"Paris"` | -0.105 | -0.916 | $0.1 \times [(-0.105)-(-0.916)] = \mathbf{+0.0811}$ |
 -->


<!-- TRL > 0.29 parameter change

model_adapter_name (str, optional) — Name of the train target PEFT adapter, when using LoRA with multiple adapters. Only the default adapter will be supported going forward.
This parameter is deprecated and will be removed in version 0.29.0. Only the default adapter will be supported going forward.

ref_adapter_name (str, optional) — Name of the reference PEFT adapter, when using LoRA with multiple adapters. If you used it to resume training an adapter, you won’t need this argument anymore in the next version and can rely on the trainer. For now, it is still the only supported way to do this.
This parameter is deprecated and will be removed in version 0.29.0. If you used it to resume training an adapter, you won’t need this argument anymore in the next version and can rely on the trainer. For now, it is still the only supported way to do this.

https://huggingface.co/docs/trl/v0.28.0/en/dpo_trainer?utm_source=chatgpt.com
 -->


In [1]:
# Work around a TRL 0.24.0 + Transformers 5.5.0 bug where prompt-only apply_chat_template() returns a BatchEncoding, causing TRL to miscompute the prompt length (e.g., len(prompt_ids) == 2) and incorrectly mask only the first few prompt tokens.
# SFT caompatibe != TRL 0.24.0 & Transformers 5.5 -> prompt-only apply_chat_template() error reported

%pip install --upgrade --no-cache-dir transformers trl datasets peft accelerate bitsandbytes safetensors
!pip install --upgrade "torchao>0.16.0" # PEFT requirement

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 427.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 226.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 317.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 49.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Colab_Notebooks') # change directory to the current working directory

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import login
notebook_login()

login(token="[HF Token]")

In [4]:
import pandas as pd
import json

DPO_TRAIN_DATA_LOAD_PATH = "data_preference_train_train_only/bridge_2tage__with_reference_solution_more_models/response_pairs_df.csv"
TEST_DATA_LOAD_PATH = "train_test_split/test_stepverify_labeled_0.9.json"

BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

DPO_ADAPTER_SAVE_PATH = "llama3-8b-instruct-dpo-adapter--threshold_03"
DPO_LOCAL_ADAPTER_DIR = os.path.join("/content", DPO_ADAPTER_SAVE_PATH) # save in the current colab loacl disk ( circumvent google drive i/o limit )
DRIVE_ROOT_DIR = "/content/drive/My Drive/Colab_Notebooks" # current notebook directory in the google drive
DPO_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_SAVE_PATH)    # save in the ADAPTER directory in the google drive

SFT_ADAPTER_PATH = "llama3-8b-instruct-sft-adapter" # Adapter name # DPO (pi_theat, pi_ref)
SFT_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, SFT_ADAPTER_PATH)

SEED = 42

#### Load data

In [5]:
pref_data_df = pd.read_csv(DPO_TRAIN_DATA_LOAD_PATH)
test_data = json.load(open(TEST_DATA_LOAD_PATH, "r"))
# test_data = pd.DataFrame(test_data).rename(columns={"student_incorrect_solution": "student_mistake"}).to_dict(orient="records") # change the key name to match pref_data_format
for row in test_data:
    row["student_mistake"] = row.pop("student_incorrect_solution")

In [6]:
def get_data_threshold(df, threshold=1):
  """
    threshold parameter determines the percentage of top-scoring response pairs to keep within each error category.
    data: json type
    return: json type
  """
  top_thresholod_response_pairs_list = []

  for category, group in df.groupby('error_category'):
      group_sorted = group.sort_values(by='score_difference', ascending=False)
      n_top = int(len(group_sorted) * threshold)
      n_top = max(n_top, 1)  # at least one example per each error category
      top_threshold_group = group_sorted.head(n_top)
      top_thresholod_response_pairs_list.append(top_threshold_group)

  top_thresholod_response_pairs_df = pd.concat(top_thresholod_response_pairs_list, ignore_index=True)

  return top_thresholod_response_pairs_df.to_dict(orient="records")


In [7]:
pref_data = get_data_threshold(pref_data_df, threshold=0.3)

#### train val split

In [8]:
from sklearn.model_selection import train_test_split
import pandas as pd


def split_train_test(data, stratify_col="error_category", test_size=0.3, random_state=SEED):
    # stratify에 사용할 label
    stratify_labels = [item[stratify_col] for item in data]
    indices = list(range(len(data)))

    train_data, val_data, train_idx, val_idx = train_test_split(
        data,
        indices,
        test_size=0.20,
        stratify = [item["error_category"] for item in data],
        random_state=SEED,
        shuffle=True,
    )

    return train_data, val_data, train_idx, val_idx

In [9]:
train_data, val_data, train_idx, val_idx = split_train_test(pref_data)

In [10]:
print(
    f"Before split length: {len(pref_data)} | "
    f"label types: {len(set(item['error_category'] for item in pref_data))} | "
    f"label counts: | "
    f"Mean normalized score difference:  "
    f"{pd.Series([item['normalized_score_difference'] for item in pref_data]).mean():.4f} \n"
)

all_counts = pd.Series(
    [item["error_category"] for item in pref_data]
).value_counts()

print(all_counts)

print("-" * 100)

print(
    f"Total Train Length: {len(train_data)} | "
    f"label types: {len(set(item['error_category'] for item in train_data))} | "
    f"Train label counts: | "
    f"Mean normalized score difference: "
    f"{pd.Series([item['normalized_score_difference'] for item in train_data]).mean():.4f} \n"
)

train_counts = pd.Series(
    [item["error_category"] for item in train_data]
).value_counts()

print(train_counts)

print("-" * 100)

print(
    f"Total Validation Length: {len(val_data)} | "
    f"label types: {len(set(item['error_category'] for item in val_data))} | "
    f"Validation label counts: | "
    f"Mean normalized score difference: "
    f"{pd.Series([item['normalized_score_difference'] for item in val_data]).mean():.4f} \n"
)

val_counts = pd.Series(
    [item["error_category"] for item in val_data]
).value_counts()

print(val_counts)

print("-" * 100)

print(
    f"Total Test Length: {len(test_data)} | "
    f"label types: {len(set(item['error_category'] for item in test_data))} | "
    # f"Mean normalized score difference: "
    # f"{pd.Series([item['normalized_score_difference'] for item in test_data]).mean():.4f} | "
    f"Test label counts:\n"
)

test_counts = pd.Series(
    [item["error_category"] for item in test_data]
).value_counts()

print(test_counts)

Before split length: 7286 | label types: 7 | label counts: | Mean normalized score difference:  0.2247 

misunderstanding_of_a_question                     2089
extra_quantity_or_missing_quantity                 1762
missing_wrong_factual_knowledge                    1019
calculation_error_easily_solved_by_a_calculator     909
none_of_the_above                                   631
reached_correct_solution_but_proceeded_further      515
unit_conversion_error                               361
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Total Train Length: 5828 | label types: 7 | Train label counts: | Mean normalized score difference: 0.2244 

misunderstanding_of_a_question                     1671
extra_quantity_or_missing_quantity                 1409
missing_wrong_factual_knowledge                     815
calculation_error_easily_solved_by_a_calculator     727
none_of_the_above                          

#### Format and load dataset

In [11]:
from datasets import Dataset

SYSTEM_TEMPLATE = """You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.
"""

USER_TEMPLATE = """
### Problem:
{problem}

### Student's response:
{student}
"""


def make_dpo_train_val_dataset(data):
    dataset = Dataset.from_list([
        {
            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM_TEMPLATE,
                },
                {
                    "role": "user",
                    "content": USER_TEMPLATE.format(
                        problem=str(row["problem"]),
                        student=str(row["student_mistake"]),
                    ),
                },
            ],

            "chosen": [
                {
                    "role": "assistant",
                    "content": str(row["chosen_response"]),
                }
            ],

            "rejected": [
                {
                    "role": "assistant",
                    "content": str(row["rejected_response"]),
                }
            ],
        }
        for row in data
    ])

    return dataset


def make_dpo_test_dataset(data):
    dataset = Dataset.from_list([
        {
            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM_TEMPLATE,
                },
                {
                    "role": "user",
                    "content": USER_TEMPLATE.format(
                        problem=str(row["problem"]),
                        student=str(row["student_mistake"]),
                    ),
                },
            ],
            "completion": [
              {
                  "role": "assistant",
                  "content": "(" + str(row['dialog_history'][0]['pedagogy']) + ")" + str(row['dialog_history'][0]['text']),
              }
          ],
        }
        for row in data
    ])

    return dataset

In [12]:
train_ds, val_ds = make_dpo_train_val_dataset(train_data), make_dpo_train_val_dataset(val_data)
test_ds = make_dpo_test_dataset(test_data)

print(f"Train: {len(train_ds)} ")
print(f"Valid: {len(val_ds)} ")
print(f"Test:  {len(test_ds)} ")

Train: 5828 
Valid: 1458 
Test:  298 


#### Load PEFT model and SFT adapter
- https://huggingface.co/docs/transformers/ko/peft

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel #, prepare_model_for_kbit_training


# load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
base_model.config.use_cache = False

# load pretrained tokenizer
dpo_tokenizer = AutoTokenizer.from_pretrained(
    SFT_DRIVE_MODEL_DIR,
    clean_up_tokenization_specs=False
)

# Load SFT LoRA & its adapter
dpo_model = PeftModel.from_pretrained(
    base_model,
    SFT_DRIVE_MODEL_DIR,
    is_trainable=True, # SFT Adapter parameters will be further updated during DPO
    adapter_name="default", # TRL > 0.29 : Set a default adapter, and then DPOTrainer will copy the adapter and set it as the ref adapter. model.set_adapter / adpater_name / config( model_adapter_name / ref_adapter_name ) -> no needed from 0.29
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

#### DPO Config

In [14]:
from trl import DPOConfig, DPOTrainer


dpo_training_args = DPOConfig(
    output_dir=DPO_DRIVE_MODEL_DIR,

    # DPO
    beta=0.1, # controls the deviation from the reference model. ( Default : 0.1 )

    precompute_ref_log_probs=True, # True : reduces the repetitive log_prob computations during reference forward across epochs
    precompute_ref_batch_size=4,


    # Batch
    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    # gradient_accumulation_steps=4,

    max_length=1024,

    # Optimization
    learning_rate=5e-6, # set it less than SFT learning rate. (e.g., sft 2e-5 -> dop 5e-6)
    lr_scheduler_type="cosine",
    warmup_steps=0.01,

    bf16=True,
    train_sampling_strategy="random",


    # Evaluation / logging
    eval_strategy="steps",
    eval_steps=200,

    logging_steps=25,
    save_strategy="steps",
    save_steps=200,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",


    # Gradient checkpointing
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False,},
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_training_args,
    train_dataset=train_ds, #.select(range(100)),
    eval_dataset=val_ds, #.select(range(100)),
    processing_class=dpo_tokenizer,
)

Tokenizing train dataset:   0%|          | 0/5828 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/5828 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1458 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/1458 [00:00<?, ? examples/s]

Computing reference log probs for train dataset:   0%|          | 0/1457 [00:00<?, ?it/s]

...

Flattening the indices:   0%|          | 0/5828 [00:00<?, ? examples/s]

Computing reference log probs for eval dataset:   0%|          | 0/365 [00:00<?, ?it/s]

Caching reference log probs for eval dataset:   0%|          | 0/1458 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1458 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1458 [00:00<?, ? examples/s]

In [15]:
# Cehck Active Adapters
all_adapters = set(dpo_model.peft_config.keys())
active = dpo_model.active_adapters # Active adapter

active_adapters = set(active if isinstance(active, (list, tuple, set)) else [active])
inactive_adapters = all_adapters - active_adapters

print("loaded adapters")
print("All.     :", sorted(all_adapters))
print("Active   :", sorted(active_adapters))
print("Inactive :", sorted(inactive_adapters))

loaded adapters
All.     : ['default', 'ref']
Active   : ['default']
Inactive : ['ref']


In [16]:
dpo_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
200,0.547017,0.527585,1.505201,2158924.000000,0.164160,0.068353,0.593152,0.300037,-0.376080,0.730978,0.676118,-117.012536,-119.491189
400,0.438436,0.501172,1.429330,4321151.000000,0.102075,0.000344,0.597904,0.492121,-0.484102,0.753397,0.976223,-115.091694,-120.571401
600,0.445720,0.486774,1.394565,6487346.000000,0.072658,-0.027068,0.597013,0.360620,-0.633060,0.762228,0.993681,-116.406704,-122.060988
800,0.438597,0.480025,1.384832,8637204.000000,0.036240,-0.061520,0.594352,0.213916,-0.779190,0.764266,0.993106,-117.873748,-123.522286
1000,0.382011,0.477273,1.373706,10800078.000000,0.032642,-0.064956,0.595155,0.231212,-0.784220,0.767663,1.015432,-117.700790,-123.572583
1095,0.413134,0.477633,1.373113,11822655.000000,0.032813,-0.064706,0.594931,0.224157,-0.790938,0.764266,1.015095,-117.771339,-123.639768


TrainOutput(global_step=1095, training_loss=0.45920530084061295, metrics={'train_runtime': 3601.7602, 'train_samples_per_second': 4.854, 'train_steps_per_second': 0.304, 'total_flos': 6.723916424968274e+17, 'train_loss': 0.45920530084061295, 'epoch': 3.0})

#### Save model

In [17]:
# https://chatgpt.com/s/t_6a8e3e8e08748191955c27ce04625c4d

import os
import shutil

# 1. Save LoRA adapter
# trainer.save_model(LOCAL_ADAPTER_DIR)
dpo_trainer.model.save_pretrained(DPO_DRIVE_MODEL_DIR)

# 2. Save Tokenizer
dpo_tokenizer.save_pretrained(DPO_DRIVE_MODEL_DIR)

print("Saved to Google Drive:", DPO_DRIVE_MODEL_DIR)

Saved to Google Drive: /content/drive/My Drive/Colab_Notebooks/llama3-8b-instruct-dpo-adapter--threshold_03


----

#### Inference

In [18]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel, PeftConfig
# import torch

# # Read adapter config
# peft_config = PeftConfig.from_pretrained(
#     DRIVE_MODEL_DIR # DPO_ADAPTER_DIR
# )

# base_model_id = peft_config.base_model_name_or_path
# print("Base model:", base_model_id)

# # load base model
# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     dtype=torch.bfloat16,
#     device_map={"": 0},
# )

# # load base toeknizer
# base_tokenizer = AutoTokenizer.from_pretrained(
#     base_model_id,
# )


# # load DPO model ( basemodel + DPO adapter )
# dpo_model = PeftModel.from_pretrained(
#     base_model,
#     DRIVE_MODEL_DIR,
#     # inference이므로 frozen
#     is_trainable=False,
#     adapter_name="default",
# )

# # load dpo tokenizer
# dpo_tokenizer = AutoTokenizer.from_pretrained(
#     DRIVE_MODEL_DIR,
#     clean_up_tokenization_spaces=False,
# )

# if dpo_tokenizer.pad_token is None:
#     dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

# dpo_tokenizer.padding_side = "left"

# base_model.config.pad_token_id = (
#     dpo_tokenizer.pad_token_id
# )


In [19]:
# sft_peft_config = PeftConfig.from_pretrained(
#     DPO_ADAPTER_PATH_2
# )

# base_model_id_2 = peft_config.base_model_name_or_path
# print("Base model:", base_model_id_2)

# # load base model
# base_model_2 = AutoModelForCausalLM.from_pretrained(
#     base_model_id_2,
#     dtype=torch.bfloat16,
#     device_map={"": 0},
# )


# dpo_model_2 = PeftModel.from_pretrained(
#     base_model,
#     DPO_ADAPTER_PATH_2,
#     # inference이므로 frozen
#     is_trainable=False,
#     adapter_name="default",
# )

# dpo_tokenizer_2 = AutoTokenizer.from_pretrained(
#     DPO_ADAPTER_PATH_2,
#     clean_up_tokenization_spaces=False
# )

In [20]:
# # SFT model
# SFT_ADAPTER_PATH = "llama3-8b-instruct-sft-adapter" # Adapter name
# SFT_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, ADAPTER_PATH)    # save in the ADAPTER directory in the google drive

# sft_model = AutoModelForCausalLM.from_pretrained(
#     SFT_DRIVE_MODEL_DIR,
#     dtype=torch.bfloat16,
#     device_map="auto")

# sft_tokenizer = AutoTokenizer.from_pretrained(
#     SFT_DRIVE_MODEL_DIR,
#     clean_up_tokenization_spaces=False,
#     )


In [21]:
# from transformers import pipeline

# def compare_responses(data):
#   ####################################
#   # Fine-tuned Model Response
#   ####################################
#   prompt = dpo_tokenizer.apply_chat_template(
#       data["prompt"],
#       tokenize=False,
#       add_generation_prompt=True) # 현재 Transformers의 text-generation pipeline은 messages 형태를 직접 받을 수 있고, chat template도 pipeline이 알아서 적용합니다 https://github.com/huggingface/transformers/blob/main/src/transformers/pipelines/text_generation.py?utm_source=chatgpt.com

#   base_pipe = pipeline(
#       task="text-generation",
#       model=base_model,
#       tokenizer=dpo_tokenizer,
#       return_full_text=False)

#   sft_pipe = pipeline(
#       task="text-generation",
#       model=sft_model,
#       tokenizer=sft_tokenizer,
#       return_full_text=False)

#   dpo_pipe = pipeline(
#       task="text-generation",
#       model=dpo_model,
#       tokenizer=dpo_tokenizer,
#       return_full_text=False)

#   dpo_pipe_2 = pipeline(
#       task="text-generation",
#       model=dpo_model_2,
#       tokenizer=dpo_tokenizer_2,
#       return_full_text=False)

#   # apply_chat_template
#   # add_generation_prompt=True is needed for the model to generate the next tokens as trained : https://huggingface.co/docs/transformers/chat_templating

#   print("=" * 200)
#   print(" [ Prompt ] \n")
#   print(prompt)

#   print("=" * 200)

#   print("[ Ground Truth ] ")
#   ground_truth = data['completion'][0]['content']
#   print(ground_truth)

#   print("=" * 200)

#   print("[ Base-model Response ] ")
#   base_model_response = base_pipe(prompt)[0]["generated_text"]
#   print(base_model_response)

#   print("=" * 200)

#   print("[ SFT Response ]")
#   sft_model_response = sft_pipe(prompt)[0]["generated_text"]
#   print(sft_model_response)

#   print("=" * 200)

#   print("[ DPO Response (top 30%)]")
#   dpo_model_response = dpo_pipe(prompt)[0]["generated_text"]
#   print(dpo_model_response)

#   print("=" * 200)

#   print("[ DPO Response (top 20%)]")
#   dpo_model_response = dpo_pipe_2(prompt)[0]["generated_text"]
#   print(dpo_model_response)

#   return {
#       "prompt": prompt,
#       "ground_truth": ground_truth,
#       "base_model_response": base_model_response,
#       "sft_model_response": sft_model_response,
#       "dpo_model_response": dpo_model_response
#   }


# results = []
# for idx, test in enumerate(test_ds):
#   result = compare_responses(test)
#   results.append(result)
#   print("\n\n\n")


# responses_df = pd.DataFrame(results)
# responses_df.to_csv("base_sft_dpo_responses.csv")
# print("saved_to ", "base_sft_dpo_responses.csv")
